# Train Part 1 (ML)
Notebook này là phần ML được tách ra từ `train_part1_features.ipynb`.
Mục tiêu: trích xuất feature HRV/engineered và lưu `features_hrv.csv`.


In [ ]:
# 1) Setup & Imports (ML feature extraction only)
from pathlib import Path

import numpy as np
import pandas as pd

# ---- Project paths ----
RAW_FOLDER = Path(r"C:/Users/buck/Napplee/StressClassification/data/concatenated/day_scenarios/")
FEATURE_FILE = Path(r"C:/Users/buck/Napplee/StressClassification/data/features/features_hrv.csv")
FEATURE_FILE.parent.mkdir(parents=True, exist_ok=True)

# ---- Extraction config ----
WINDOW_SEC = 30  # 15 minutes
MIN_RR_COUNT = 5
MIN_WINDOW_DURATION_SEC = 20.0
MIN_HR_BPM = 40.0
MAX_HR_BPM = 190.0

print(f"Raw folder: {RAW_FOLDER}")
print(f"Output feature file: {FEATURE_FILE}")

Raw folder: C:\Users\buck\Napplee\StressClassification\data\concatenated\day_scenarios
Output feature file: C:\Users\buck\Napplee\StressClassification\data\features\features_hrv.csv


In [29]:
def clean_waveform_group(g):
    g = g.sort_values("Time").drop_duplicates("Time")
    g = g[g["Time"].notna() & (g["Time"] >= 0)]
    g = g[g["Time"].diff().fillna(1) > 0]
    g["Peak"] = pd.to_numeric(g["Peak"], errors="coerce")
    v = pd.to_numeric(g["Voltage"], errors="coerce")
    q_low, q_high = np.nanquantile(v, [0.005, 0.995])
    g["Voltage"] = v.clip(lower=q_low, upper=q_high)
    return g.dropna(subset=["Time", "Voltage", "Peak"] )

def safe_skew(x):
    return float(pd.Series(x).skew()) if len(x) >= 3 else np.nan

def safe_kurt(x):
    return float(pd.Series(x).kurt()) if len(x) >= 4 else np.nan

def rr_frequency_features(rr_ms):
    if len(rr_ms) < 4:
        return (np.nan, np.nan, np.nan, np.nan)
    rr_s = rr_ms / 1000.0
    t = np.cumsum(rr_s) - rr_s[0]
    if t[-1] <= 0:
        return (np.nan, np.nan, np.nan, np.nan)
    fs = 4.0
    t_uniform = np.arange(0, t[-1], 1 / fs)
    if len(t_uniform) < 8:
        return (np.nan, np.nan, np.nan, np.nan)
    rr_interp = np.interp(t_uniform, t, rr_ms)
    rr_detrended = rr_interp - np.nanmean(rr_interp)
    fft_vals = np.fft.rfft(rr_detrended)
    freqs = np.fft.rfftfreq(len(rr_detrended), d=1 / fs)
    psd = (np.abs(fft_vals) ** 2) / max(len(rr_detrended), 1)
    lf_mask = (freqs >= 0.04) & (freqs < 0.15)
    hf_mask = (freqs >= 0.15) & (freqs <= 0.40)
    total_mask = (freqs >= 0.04) & (freqs <= 0.40)
    lf_power = float(np.trapezoid(psd[lf_mask], freqs[lf_mask])) if np.any(lf_mask) else np.nan
    hf_power = float(np.trapezoid(psd[hf_mask], freqs[hf_mask])) if np.any(hf_mask) else np.nan
    total_power = float(np.trapezoid(psd[total_mask], freqs[total_mask])) if np.any(total_mask) else np.nan
    lf_hf_ratio = (lf_power / hf_power) if (pd.notna(lf_power) and pd.notna(hf_power) and hf_power > 0) else np.nan
    return lf_power, hf_power, lf_hf_ratio, total_power

def extract_features_from_window(g, window_id=0, source_file="raw_file", window_start=None, window_end=None):
    g = clean_waveform_group(g)
    duration_sec = float(g["Time"].max() - g["Time"].min()) if len(g) else 0.0
    if len(g) < 20 or duration_sec < MIN_WINDOW_DURATION_SEC:
        return None
    peak_times = g.loc[g["Peak"] == 3, "Time"].values
    if len(peak_times) < 4:
        return None
    rr_raw = np.diff(peak_times) * 1000.0  # ms
    rr_ms = rr_raw[(rr_raw >= 300.0) & (rr_raw <= 2000.0)]
    rr_valid_ratio = float(len(rr_ms) / max(len(rr_raw), 1))
    if len(rr_ms) < MIN_RR_COUNT:
        return None
    rmssd = np.sqrt(np.mean(np.diff(rr_ms) ** 2)) if len(rr_ms) >= 2 else np.nan
    pnn50 = float(np.mean(np.abs(np.diff(rr_ms)) > 50.0) * 100.0) if len(rr_ms) >= 2 else np.nan
    mean_rr_ms = float(np.mean(rr_ms))
    sdnn_ms = float(np.std(rr_ms, ddof=1)) if len(rr_ms) > 1 else np.nan
    hr_bpm = 60000.0 / mean_rr_ms if mean_rr_ms > 0 else np.nan
    n_peaks = int(np.sum(g["Peak"] == 3))
    peak_rate = n_peaks / duration_sec if duration_sec > 0 else np.nan
    lf_power, hf_power, lf_hf_ratio, total_power = rr_frequency_features(rr_ms)
    label_val = g["label"].mode().iloc[0] if "label" in g.columns and not g["label"].isnull().all() else np.nan
    return {
        "source_file": source_file,
        "window_id": window_id,
        "window_start": window_start,
        "window_end": window_end,
        "label": label_val,
        "mean_rr_ms": mean_rr_ms,
        "median_rr_ms": float(np.median(rr_ms)),
        "sdnn_ms": sdnn_ms,
        "rmssd_ms": float(rmssd),
        "pnn50": float(pnn50),
        "heart_rate_bpm": float(hr_bpm),
        "n_peaks": n_peaks,
        "duration_sec": duration_sec,
        "peak_rate_per_sec": float(peak_rate),
        "rr_valid_ratio": rr_valid_ratio,
        "voltage_mean": float(g["Voltage"].mean()),
        "voltage_std": float(g["Voltage"].std(ddof=1)) if len(g) > 1 else np.nan,
        "voltage_min": float(g["Voltage"].min()),
        "voltage_max": float(g["Voltage"].max()),
        "voltage_skew": safe_skew(g["Voltage"].values),
        "voltage_kurtosis": safe_kurt(g["Voltage"].values),
        "lf_power": lf_power,
        "hf_power": hf_power,
        "lf_hf_ratio": lf_hf_ratio,
        "total_power": total_power,
    }

In [30]:
# ---- MAIN ----

all_feature_rows = []

for csv_path in RAW_FOLDER.glob("*.csv"):
    try:
        df = pd.read_csv(csv_path)
        if not all(col in df.columns for col in ["Time", "Voltage", "Peak"]):
            print(f"Bỏ qua {csv_path.name} vì thiếu cột bắt buộc!")
            continue
        df = df.sort_values("Time").reset_index(drop=True)
        min_time = df["Time"].min()
        max_time = df["Time"].max()
        n_windows = int(np.ceil((max_time - min_time) / WINDOW_SEC))
        for i in range(n_windows):
            start = min_time + i * WINDOW_SEC
            end = start + WINDOW_SEC
            win_df = df[(df["Time"] >= start) & (df["Time"] < end)].copy()
            if win_df.empty:
                continue
            row = extract_features_from_window(
                win_df,
                window_id=i,
                source_file=csv_path.name,
                window_start=start,
                window_end=end
            )
            if row is not None:
                all_feature_rows.append(row)
    except Exception as ex:
        print(f"Lỗi đọc {csv_path}: {ex}")

features_df = pd.DataFrame(all_feature_rows)
if features_df.empty:
    raise ValueError("Không trích xuất được feature từ bất kỳ file nào!")

# ---- (OPTIONAL) Lọc chất lượng đầu ra ----
features_df = features_df[
    (features_df["duration_sec"] >= MIN_WINDOW_DURATION_SEC) &
    (features_df["heart_rate_bpm"].between(MIN_HR_BPM, MAX_HR_BPM))
]
features_df = features_df.reset_index(drop=True)

features_df.to_csv(FEATURE_FILE, index=False)
print(f"Đã trích xuất và lưu {len(features_df)} window vào {FEATURE_FILE}")
print("Phân bố label:")
print(features_df["label"].value_counts(dropna=False))
print(features_df.head())

Đã trích xuất và lưu 720 window vào C:\Users\buck\Napplee\StressClassification\data\features\features_hrv.csv
Phân bố label:
label
0    360
1    135
3    120
2    105
Name: count, dtype: int64
                   source_file  window_id  window_start  window_end  label  \
0  day_custom_segments_001.csv          0           0.0      1800.0      0   
1  day_custom_segments_001.csv          1        1800.0      3600.0      0   
2  day_custom_segments_001.csv          2        3600.0      5400.0      0   
3  day_custom_segments_001.csv          3        5400.0      7200.0      0   
4  day_custom_segments_001.csv          4        7200.0      9000.0      0   

   mean_rr_ms  median_rr_ms    sdnn_ms   rmssd_ms      pnn50  ...  \
0  711.804354     707.03125  85.078109  63.927008  36.406806  ...   
1  712.470000     703.12500  95.205423  69.283021  36.079208  ...   
2  688.687428     679.68750  86.204835  62.725454  31.750287  ...   
3  687.388094     679.68750  82.480155  62.489784  31.677493  